In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
import geopandas as gpd
import sys
from scipy import stats

sys.path.insert(1, '../../ProcessEvents')
from config import CATCHMENTS
from Functions import *

sys.path.insert(1, '../RandomForest')
from PrepDataFunctions import *

## ------------------------------------------------------------ ###
### Get rainfall events
    And add extra variables
## ------------------------------------------------------------ ###

In [ ]:
df = prep_for_modelling(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/all_catchments_with_topo_and_temporal.pkl", 
                        remove_events_with_zero_flooding = False)
df = df[df['mean_sm_2_before_event'].notna()]
df =df[df['start_year']>1990]

## ------------------------------------------------------------ ###
### Get spatial data and add regional geometries to rainfall data
## ------------------------------------------------------------ ###

In [ ]:
df  = add_geometries(df, CATCHMENTS)

## ------------------------------------------------------------ ###
### Add column defining which of 3 temporal periods an event is within
## ------------------------------------------------------------ ###

In [ ]:
conditions = [df["start_year"].between(1990, 2020),df["start_year"].between(2020, 2050),
    df["start_year"].between(2050, 2080),]
periods = ["1990–2020", "2020–2050", "2050–2080"]

df["period"] = np.select(conditions, periods, default=None)

## ------------------------------------------------------------ ###
### Add column defining intensity bins
## ------------------------------------------------------------ ###

In [ ]:
INT_BINS = [30, 35, 40, 45, 50, 60, 70, np.inf]
INT_LABELS = [f"{INT_BINS[i]}–{INT_BINS[i+1]}" for i in range(len(INT_BINS)-2)] + ["100+"]
df["intensity_bin"] = pd.cut(df["max_precip"], bins=INT_BINS, labels=INT_LABELS,
                                                 include_lowest=True)

LOADING_BINS = [0, 20, 40, 60, 80, 100]
LOADING_LABELS = [f"{LOADING_BINS[i]}–{LOADING_BINS[i+1]}" for i in range(len(LOADING_BINS)-1)] 
df["loading_bin"] = pd.cut(df["d50_n"], bins=LOADING_BINS, labels=LOADING_LABELS, 
                                               include_lowest=True)

PEAK_SHARE_BINS = [0, 0.20, 0.40, 0.60, 0.80, 1]
PEAK_SHARE_LABELS = [f"{PEAK_SHARE_BINS[i]}–{PEAK_SHARE_BINS[i+1]}" for i in range(len(PEAK_SHARE_BINS)-1)] 
df["peak_share_bin"] = pd.cut(df["peak_share"], bins=PEAK_SHARE_BINS, labels=PEAK_SHARE_LABELS, 
                                               include_lowest=True)

df["peak_class"] = pd.cut(df["peak_share"], [0, 0.5, 0.8, 1.0], 
                                              labels=["spread", "mixed", "peak-dominated"], include_lowest=True)

### Plots of the mean flooded area for events of different intensities

In [ ]:
summary = (df.groupby(["period", "intensity_bin"], observed=True)["10cm_volume"].mean().unstack())
summary =summary.iloc[:,:4]
colors = plt.cm.viridis(np.linspace(0, 1, len(periods)))

# Your plot currently reverses the region order, so make the same mapping
region_colors = dict(zip(reversed(periods), colors))

fig, ax = plt.subplots(figsize=(8, 6))

for choice, color in zip(periods, colors):
    ax.plot(summary.columns,summary.loc[choice],marker="o",linewidth=3,markersize=5,color=color,label=choice)

ax.set_xlabel("Maximum rainfall intensity (mm/hr)")
ax.set_ylabel("Mean area flooded >10 cm")
ax.set_title("Mean per-event flooded area across intensity bins",pad=12)

ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.tick_params(axis="both", which="major", labelsize=12)

ax.legend(title="Time period",bbox_to_anchor=(0.05, 0.88),loc="center left",frameon=True);

In [ ]:
region_order = ["North Scotland","East Scotland","West Scotland","North West England","North East England",
                "Yorkshire and Humber","West Midlands","East Midlands","East of England","Wales",
                "South West England","South East England","London"]

In [ ]:
national_and_regional_plot(df, 'geo_region', region_order, '10cm_area', "period", periods)

In [ ]:
regional_means = (df.groupby(["geo_region", "period", "intensity_bin"], observed=True)["10cm_volume"].mean().reset_index())
national_as_mean_of_regions = (
    regional_means.groupby(["period", "intensity_bin"], observed=True)["10cm_volume"].mean().unstack())

fig, ax = plt.subplots(figsize=(8, 2))

for choice, color in zip(periods, colors):
    ax.plot(national_as_mean_of_regions.columns,national_as_mean_of_regions.loc[choice],marker="o",linewidth=3,markersize=5,
            color=color,label=choice)

ax.set_xlabel("Maximum rainfall intensity (mm/hr)")
ax.set_ylabel("Mean area flooded >10 cm")
ax.set_title("Mean of regional trends",pad=12)

ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.tick_params(axis="both", which="major", labelsize=12)

ax.legend(title="Time period",bbox_to_anchor=(0.05, 0.88),loc="center left",frameon=True);

## Split by event length

In [ ]:
fig, ax = plt.subplots(ncols= 3, figsize=(15, 4), sharex=True, sharey=True)
ax_num=0
lengths = {'short': (0, 10),'mid': (10, 30), 'long':  (30, 200),}
colors = plt.cm.viridis(np.linspace(0, 1, len(periods)))

for label, (y0, y1) in lengths.items():
    mask = (df['length_of_active_rain'] >= y0) & (df['length_of_active_rain'] <= y1)
    this_length_df = df.loc[mask]
    summary = (this_length_df.groupby(["period", "intensity_bin"], observed=True)["10cm_volume"].mean().unstack())
    
    # Your plot currently reverses the region order, so make the same mapping
    region_colors = dict(zip(periods, colors))

    for region, color in zip(periods, colors):
        ax[ax_num].plot(summary.columns,summary.loc[region],marker="o",linewidth=3,markersize=5,color=color,label=region)

    ax[ax_num].set_xlabel("Maximum rainfall intensity (mm/hr)")
    if ax_num == 0:
        ax[ax_num].set_ylabel("Mean area flooded >10 cm")
    #ax[ax_num].set_title("Mean per-event flooded area across intensity bins",
    #             fontsize=20, pad=12)
    
    ax[ax_num].set_title(label)
    ax[ax_num].grid(axis="y", alpha=0.25)
    ax[ax_num].spines["top"].set_visible(False)
    ax[ax_num].spines["right"].set_visible(False)
    ax[ax_num].tick_params(axis="both", which="major", labelsize=12)
    
    if ax_num == 2:
        ax[ax_num].legend(title="Region",bbox_to_anchor=(1.02, 0.5),loc="center left",frameon=False,fontsize=15)
    
    ax_num =ax_num+1
    
fig.tight_layout()

In [ ]:
trend_results = []
period_numeric = {'1990–2020': 0, '2020–2050': 1, '2050–2080': 2}  # adjust to your actual labels

for region in region_order:
    # Get data just this region
    sub = df[df['geo_region'] == region]
    # For each intensity bin
    for ibin in sub['intensity_bin'].cat.categories:
        bsub = sub[sub['intensity_bin'] == ibin].copy()
        bsub['period_num'] = bsub['period'].map(period_numeric)
        if bsub['period_num'].nunique() < 2:
            continue
        slope, intercept, r, p, se = stats.linregress(bsub['period_num'], bsub['10cm_volume'])
        trend_results.append({'region': region, 'intensity_bin': ibin, 'slope': slope, 'p_value': p, "se": se})

trend_df = pd.DataFrame(trend_results)
# trend_df[trend_df['p_value'] < 0.05]  # which region/bin combinations show a credible trend

In [ ]:
# Pivot trend_df into a region x intensity_bin grid of slopes
slope_grid = trend_df.pivot(index='region', columns='intensity_bin', values='slope')
pval_grid = trend_df.pivot(index='region', columns='intensity_bin', values='p_value')

# Order rows consistently (e.g. by overall mean slope, or your existing region_order)
slope_grid = slope_grid.reindex(region_order)
pval_grid = pval_grid.reindex(region_order)

fig, ax = plt.subplots(figsize=(8, 9))

vmax = np.nanmax(np.abs(slope_grid.values))
im = ax.imshow(slope_grid.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

# Mark significant cells with an asterisk
for i in range(slope_grid.shape[0]):
    for j in range(slope_grid.shape[1]):
        p = pval_grid.values[i, j]
        if not np.isnan(p) and p < 0.05:
            ax.text(j, i, '*', ha='center', va='center', color='black', fontsize=14, fontweight='bold')

ax.set_xticks(range(len(slope_grid.columns)))
ax.set_xticklabels(slope_grid.columns)
ax.set_yticks(range(len(slope_grid.index)))
ax.set_yticklabels(slope_grid.index)
ax.set_xlabel('Intensity bin')
plt.colorbar(im, ax=ax, label='Trend in mean severity per period-step')
ax.set_title('Regional trends by intensity bin\n(* = p < 0.05)')
plt.tight_layout()
plt.show()

In [ ]:
# Reconcile column naming between trend_df ('region') and the main dataframe ('geo_region')
region_centroids = (
    df.groupby('geo_region')[['x_coord', 'y_coord']]
    .mean().reset_index().rename(columns={'geo_region': 'region'}))

trend_with_coords = trend_df.merge(region_centroids, on='region', how='left')

intensity_bins_available = trend_with_coords['intensity_bin'].cat.categories if hasattr(trend_with_coords['intensity_bin'], 'cat') else trend_with_coords['intensity_bin'].unique()

fig, axes = plt.subplots(1, len(intensity_bins_available), figsize=(5*len(intensity_bins_available), 5), sharex=True, sharey=True)

vmax = np.nanmax(np.abs(trend_with_coords['slope']))

for ax, ibin in zip(axes, intensity_bins_available):
    sub = trend_with_coords[trend_with_coords['intensity_bin'] == ibin]
    edgewidths = np.where(sub['p_value'] < 0.05, 2.0, 0.3)  # use p_adjusted if you've applied the FDR correction

    sc = ax.scatter(sub['x_coord'], sub['y_coord'], c=sub['slope'], cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax, s=180, edgecolor='k', linewidth=edgewidths)
    ax.set_title(ibin)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(sc, ax=axes, label='Trend in mean severity', shrink=0.7)
fig.suptitle('Spatial pattern of regional trends, by intensity bin\n(thick outline = significant after FDR correction)')
plt.show()